In [4]:
import sys
!{sys.executable} -m pip install pandas numpy sqlalchemy pyodbc pymongo

  Using cached dnspython-2.8.0-py3-none-any.whl.metadata (5.7 kB)
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ------------------------ --------------- 6.0/9.9 MB 44.6 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 30.0 MB/s  0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 12.6/12.6 MB 97.8 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 58.9 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 53.0 MB/s  0:00:00
Using cached dnspython-2.8.0-py3-none-any.whl (331 kB)

   ---------------------------------------- 0/8 [tzdata]
   ---------------------------------------- 0/8 [tzdata]
   ---------------------------------------- 0/8 [tzdata]
   ---------------------------------------- 0/8 [tzdata

# Fase 1 — ETL: Limpieza, Feature Engineering y Carga a SQL Server

Pasos 1.2, 1.3 y 1.4 del proyecto.

## Paso 1.2 — Limpieza Inicial

In [5]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from pymongo import MongoClient

#### Carga de Datos (Ingesta)

In [ ]:
# ============================================================================
# BLOQUE 2: CARGA, LECTURA E INSPECCIÓN DE TIPOS
# ============================================================================
print(">>> Paso 1: Cargando el dataset original desde la ruta del repositorio...")

df = pd.read_csv(
    r'C:\Users\HP\OneDrive\proyecto-personas-desaparecidas\data\raw\personasdesaparecidas.csv',
    sep=';',                    # El archivo utiliza punto y coma como separador
    encoding='utf-8-sig',       # Elimina el BOM inicial del archivo
    na_values=['NO_APLICA'],    # Mapea textos 'NO_APLICA' directamente a NaN
    parse_dates=['fecha_desaparicion', 'fecha_denuncia', 'fecha_localizacion'],
    dayfirst=True,              # Soporte para formato de fechas D/M/YYYY
    low_memory=False,
)

# Quitar columna fantasma "Unnamed" generada por el punto y coma de cierre en el CSV
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
print(df.shape)
print(df.info())



>>> Paso 1: Cargando el dataset original desde la ruta del repositorio...
(75680, 27)
<class 'pandas.DataFrame'>
RangeIndex: 75680 entries, 0 to 75679
Data columns (total 27 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   fecha_desaparicion                 75680 non-null  datetime64[us]
 1   fecha_denuncia                     75680 non-null  datetime64[us]
 2   fecha_conocimiento                 7490 non-null   str           
 3   zona                               75680 non-null  str           
 4   distrito                           75680 non-null  str           
 5   circuito                           75680 non-null  str           
 6   subcircuito                        75680 non-null  str           
 7   codigo_provincia                   75680 non-null  int64         
 8   provincia                          75680 non-null  str           
 9   codigo_canton                 

#### Limpieza y Estandarización Canónica

In [ ]:
# ============================================================================
# BLOQUE 3: LIMPIEZA DE COORDENADAS Y CONVERSIÓN DE TEXTOS (FORMATO CANÓNICO)
# ============================================================================

coord_cols = ['latitud_desaparicion', 'longitud_desaparicion',
              'latitud_localizacion', 'longitud_localizacion']

for col in coord_cols:
    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Forzar casteo a numérico de variables cuantitativas
df['dias_solucion'] = pd.to_numeric(df['dias_solucion'], errors='coerce')
df['edad'] = pd.to_numeric(df['edad'], errors='coerce')

print(">>> Paso 3: Aplicando formateo canónico (MAYÚSCULAS y sin espacios)...")
columnas_texto = df.select_dtypes(include=['object']).columns

for col in columnas_texto:
    df[col] = df[col].astype(str).str.strip().str.upper()

print("Estandarización de formatos completada.")

>>> Paso 3: Aplicando formateo canónico (MAYÚSCULAS y sin espacios)...


C:\Users\HP\AppData\Local\Temp\ipykernel_2284\1643914889.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = df.select_dtypes(include=['object']).columns


Estandarización de formatos completada.


#### Tratamiento de Valores Nulos y Control de Outliers

In [22]:
# ============================================================================
# BLOQUE 4: TRATAMIENTO DE VALORES NULOS Y CONTROL DE ANOMALÍAS
# ============================================================================
print(">>> Paso 4: Iniciando auditoría y tratamiento de datos faltantes/outliers...")

# 1. Tratamiento y Corrección de la Variable de Edad
# Imputación de edad mediante la mediana estadística del contexto ecuatoriano
mediana_edad = df['edad'].median()
print(f"   * La mediana calculada para la columna 'edad' es: {mediana_edad} años.")

# Rellenar los valores nulos detectados con la mediana
df['edad'] = df['edad'].fillna(mediana_edad)

# Control de Outliers extremos en Edad (valores imposibles o inconsistentes)
registros_edad_atipica = df[(df['edad'] < 0) | (df['edad'] > 105)].shape[0]
df.loc[(df['edad'] < 0) | (df['edad'] > 105), 'edad'] = mediana_edad
print(f"   * Se detectaron y corrigieron {registros_edad_atipica} registros con edades fuera de rango.")

# 2. Remoción de Atributos Críticos con Exceso de Vacíos
# Se elimina 'fecha_conocimiento' si existe, debido a un volumen de nulos superior al 90%
if 'fecha_conocimiento' in df.columns:
    df = df.drop(columns=['fecha_conocimiento'])
    print("   * Columna 'fecha_conocimiento' eliminada del DataFrame por exceso de nulos (+90%).")

# 3. Control de Calidad sobre Coordenadas Geográficas Esenciales
# Reemplazar nulos en latitud/longitud con un valor neutro (0.0) para evitar que rompa las restricciones NOT NULL de SQL
df['latitud_desaparicion'] = df['latitud_desaparicion'].fillna(0.0)
df['longitud_desaparicion'] = df['longitud_desaparicion'].fillna(0.0)
print("   * Coordenadas geográficas aseguradas (valores nulos sustituidos por 0.0).")

# 4. Corrección Crítica de Nulos en Columnas de Motivos (Evita IntegrityError en SQL Server)
# Reemplazar valores NaN por 'SIN_DATO' y estandarizar formato canónico
df['motivo_desaparicion'] = df['motivo_desaparicion'].fillna('SIN_DATO').astype(str).str.strip().str.upper()
df['motivacion_desaparicion_observada'] = df['motivacion_desaparicion_observada'].fillna('SIN_DATO').astype(str).str.strip().str.upper()
print("   * Valores nulos en motivos y motivaciones observadas saneados como 'SIN_DATO'.")

# 5. Verificación Final del Estado de Vacíos en el DataFrame
nulos_restantes = df.isnull().sum().sum()
print(f"\n>>> Auditoría finalizada. Total de valores nulos remanentes en el DataFrame: {nulos_restantes}")

>>> Paso 4: Iniciando auditoría y tratamiento de datos faltantes/outliers...
   * La mediana calculada para la columna 'edad' es: 17.0 años.
   * Se detectaron y corrigieron 0 registros con edades fuera de rango.
   * Coordenadas geográficas aseguradas (valores nulos sustituidos por 0.0).
   * Valores nulos en motivos y motivaciones observadas saneados como 'SIN_DATO'.

>>> Auditoría finalizada. Total de valores nulos remanentes en el DataFrame: 212065


#### Ingeniería de Características e Imputación Logística

In [23]:
# ============================================================================
# BLOQUE 5: FEATURE ENGINEERING (MÉTRICA DIAS_SOLUCION)
# ============================================================================
print(">>> Paso 5: Calculando ingeniería de características (Métrica dias_solucion)...")

# Asegurar el tipo datetime para las operaciones temporales
df['fecha_desaparicion'] = pd.to_datetime(df['fecha_desaparicion'], errors='coerce')
df['fecha_localizacion'] = pd.to_datetime(df['fecha_localizacion'], errors='coerce')

# Calcular diferencia real de tiempo transcurrido en días por sistema
dias_calculados = (df['fecha_localizacion'] - df['fecha_desaparicion']).dt.days

# Rellenar vacíos de la fuente original usando la diferencia calculada
df['dias_solucion'] = df['dias_solucion'].fillna(dias_calculados)

# Control de calidad: Acotar días negativos producidos por error humano en origen
df.loc[df['dias_solucion'] < 0, 'dias_solucion'] = 0

# Auditoría final de nulos remanentes en todo el DataFrame (los de fechas y días resueltos)
nulos_restantes = df.isnull().sum().sum()
print(f"   * Métrica 'dias_solucion' calculada y controlada.")
print(f"\n>>> Auditoría finalizada. Total de valores nulos lógicos en el DataFrame: {nulos_restantes}")

>>> Paso 5: Calculando ingeniería de características (Métrica dias_solucion)...
   * Métrica 'dias_solucion' calculada y controlada.

>>> Auditoría finalizada. Total de valores nulos lógicos en el DataFrame: 212065


#### Construcción del Modelo Relacional (3FN)

In [24]:
# ============================================================================
# BLOQUE 5: AISLAMIENTO DE CATÁLOGOS DIMENSIONALES UNICOS (3FN)
# ============================================================================
print(">>> Paso 6: Aislando dimensiones en Tercera Forma Normal (3FN)...")

dim_persona = df[['sexo', 'edad', 'rango_edad', 'nacionalidad', 'etnia']].drop_duplicates().reset_index(drop=True)
dim_persona['edad'] = dim_persona['edad'].astype(int)

dim_geografia = df[['codigo_provincia', 'provincia', 'codigo_canton', 'canton', 
                    'zona', 'distrito', 'circuito', 'subcircuito']].drop_duplicates().reset_index(drop=True)
dim_geografia['codigo_provincia'] = dim_geografia['codigo_provincia'].astype(str)
dim_geografia['codigo_canton'] = dim_geografia['codigo_canton'].astype(str)

dim_motivo = df[['motivo_desaparicion', 'motivacion_desaparicion_observada']].drop_duplicates().reset_index(drop=True)

print(f"   * Dim_Persona generada: {dim_persona.shape[0]} registros únicos.")
print(f"   * Dim_Geografia generada: {dim_geografia.shape[0]} registros únicos.")
print(f"   * Dim_Motivo generada: {dim_motivo.shape[0]} registros únicos.")

>>> Paso 6: Aislando dimensiones en Tercera Forma Normal (3FN)...
   * Dim_Persona generada: 2075 registros únicos.
   * Dim_Geografia generada: 1809 registros únicos.
   * Dim_Motivo generada: 58 registros únicos.


## Carga de Datos a SQL Server

#### Persistencia Relacional en SQL Server

In [25]:
# ============================================================================
# BLOQUE 6: INYECCIÓN DE DATOS EN LA RAMA DE SQL SERVER
# ============================================================================
print(">>> Iniciando persistencia relacional...")
print(" -> Conectando a la instancia relacional de SQL Server...")
engine = create_engine('mssql+pyodbc://DESKTOP-Q2AD3LV/DB_PersonasDesaparecidas?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes')

try:
    print(" -> Inyectando catálogos maestros independientes...")
    dim_persona.to_sql('Dim_Persona', con=engine, if_exists='append', index=False)
    dim_geografia.to_sql('Dim_Geografia', con=engine, if_exists='append', index=False)
    dim_motivo.to_sql('Dim_Motivo', con=engine, if_exists='append', index=False)
    print(" -> ¡Dimensiones pobladas exitosamente!")

    print(" -> Recuperando llaves primarias autoincrementales para el mapeo...")
    df_dim_persona = pd.read_sql('SELECT id_persona, sexo, edad, rango_edad, nacionalidad, etnia FROM Dim_Persona', con=engine)
    df_dim_geografia = pd.read_sql('SELECT id_geografia, codigo_provincia, provincia, codigo_canton, canton, zona, distrito, circuito, subcircuito FROM Dim_Geografia', con=engine)
    df_dim_motivo = pd.read_sql('SELECT id_motivo, motivo_desaparicion, motivacion_desaparicion_observada FROM Dim_Motivo', con=engine)

    # Asegurar coincidencia de tipos para evitar nulos en los merge
    df['codigo_provincia'] = df['codigo_provincia'].astype(str).str.strip().str.upper()
    df['codigo_canton'] = df['codigo_canton'].astype(str).str.strip().str.upper()
    df_dim_persona['edad'] = df_dim_persona['edad'].astype(float)
    df_dim_geografia['codigo_provincia'] = df_dim_geografia['codigo_provincia'].astype(str).str.strip().str.upper()
    df_dim_geografia['codigo_canton'] = df_dim_geografia['codigo_canton'].astype(str).str.strip().str.upper()

    print(" -> Realizando el cruce relacional para heredar Llaves Foráneas (FK)...")
    fact_casos = pd.merge(df, df_dim_persona, on=['sexo', 'edad', 'rango_edad', 'nacionalidad', 'etnia'], how='left')
    fact_casos = pd.merge(fact_casos, df_dim_geografia, on=['codigo_provincia', 'provincia', 'codigo_canton', 'canton', 'zona', 'distrito', 'circuito', 'subcircuito'], how='left')
    fact_casos = pd.merge(fact_casos, df_dim_motivo, on=['motivo_desaparicion', 'motivacion_desaparicion_observada'], how='left')
    
    # Casteo a tipo compatible con enteros nulos en SQL Server
    fact_casos['dias_solucion'] = fact_casos['dias_solucion'].astype('Int64')

    columnas_fact = {
        'id_persona': 'id_persona', 'id_geografia': 'id_geografia', 'id_motivo': 'id_motivo',
        'fecha_desaparicion': 'fecha_desaparicion', 'fecha_denuncia': 'fecha_denuncia',
        'latitud_desaparicion': 'latitud_desaparicion', 'longitud_desaparicion': 'longitud_desaparicion',
        'dias_solucion': 'dias_solucion', 'situacion_actual': 'situacion_actual', 'estado_desaparecido': 'estado_desaparecido'
    }
    fact_final = fact_casos[list(columnas_fact.keys())].rename(columns=columnas_fact)

    print(f" -> Escribiendo {fact_final.shape[0]} registros en la tabla Fact_Casos...")
    fact_final.to_sql('Fact_Casos', con=engine, if_exists='append', index=False)
    print(" ¡SQL SERVER RECONSTRUIDO AL 100% CON ÉXITO!")

except Exception as e:
    print(f"\n [ERROR EN MIGRACIÓN SQL SERVER]: {e}")

>>> Iniciando persistencia relacional...
 -> Conectando a la instancia relacional de SQL Server...
 -> Inyectando catálogos maestros independientes...
 -> ¡Dimensiones pobladas exitosamente!
 -> Recuperando llaves primarias autoincrementales para el mapeo...
 -> Realizando el cruce relacional para heredar Llaves Foráneas (FK)...
 -> Escribiendo 75680 registros en la tabla Fact_Casos...
 ¡SQL SERVER RECONSTRUIDO AL 100% CON ÉXITO!


#### Persistencia NoSQL en MongoDB

In [26]:
# ============================================================================
# BLOQUE 7: INYECCIÓN DE DATOS EN LA RAMA DE MONGODB (DATA LAKE)
# ============================================================================
print("\n -> Conectando al ecosistema NoSQL de MongoDB...")
try:
    client = MongoClient('mongodb://localhost:27017/')
    db_nosql = client['DB_PersonasDesaparecidas_NoSQL']
    coleccion = db_nosql['DataLake_CasosCrudos']
    
    # Crear copia temporal y serializar objetos datetime a string para evitar errores BSON
    df_mongodb = df.copy()
    for col in df_mongodb.select_dtypes(include=['datetime64[ns]']).columns:
        df_mongodb[col] = df_mongodb[col].astype(str)
        
    data_json = df_mongodb.to_dict(orient='records')
    
    print(" -> Reemplazando e inyectando documentos estructurados en el Data Lake...")
    coleccion.drop() 
    coleccion.insert_many(data_json)
    print(" ¡MONGODB RECONSTRUIDO AL 100% CON ÉXITO!")
    print("\n========================================================================")
    print(" ¡MIGRACIÓN INTEGRAL COMPLETADA! - INFRAESTRUCTURA HÍBRIDA OPERATIVA")
    print("========================================================================")

except Exception as e:
    print(f"\n [ERROR EN INYECCIÓN MONGODB]: {e}")


 -> Conectando al ecosistema NoSQL de MongoDB...
 -> Reemplazando e inyectando documentos estructurados en el Data Lake...
 ¡MONGODB RECONSTRUIDO AL 100% CON ÉXITO!

 ¡MIGRACIÓN INTEGRAL COMPLETADA! - INFRAESTRUCTURA HÍBRIDA OPERATIVA
